# TangoFlux (T4 / L4 / G4 · 44.1 kHz Stereo Text-to-Audio & Sound Effect Generation)

Serves `declare-lab/TangoFlux` (CRPO preference-optimized 515M FluxTransformer + Flan-T5-Large + AutoencoderOobleck VAE, or `declare-lab/TangoFlux-base` via `--variant base`) on a Colab T4 GPU by default (also fast on L4/G4/A100), generating 1–30s 44.1 kHz stereo sound effects, ambient audio, and musical cues with fail-closed Bearer token auth, single & batch SFX endpoints, and Google Drive output persistence.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "tangoflux")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8000"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- install TangoFlux + stage weights & Flan-T5-Large on local NVMe SSD
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

VARIANT_REPOS = {
    "crpo": "declare-lab/TangoFlux",
    "default": "declare-lab/TangoFlux",
    "tangoflux": "declare-lab/TangoFlux",
    "base": "declare-lab/TangoFlux-base",
    "tangoflux-base": "declare-lab/TangoFlux-base",
}
TANGOFLUX_GIT_URL = "https://github.com/declare-lab/TangoFlux.git"

def tflog(msg):
    print(time.strftime("[%F %T] [tangoflux] ") + msg, flush=True)

_gpu_raw = ct_gpu_stat("name,memory.total,compute_cap").splitlines()[0]
_gpu_parts = [p.strip() for p in _gpu_raw.split(",")]
GPU_NAME = _gpu_parts[0]
_vram_tok = _gpu_parts[1].split()[0] if len(_gpu_parts) > 1 and _gpu_parts[1].strip() else ""
GPU_VRAM_MIB = int(float(_vram_tok)) if _vram_tok.replace(".", "", 1).isdigit() else 0
GPU_ARCH = (_gpu_parts[2] if len(_gpu_parts) > 2 else "7.5").replace(".", "")

RAW_VARIANT = os.environ.get("CT_TANGOFLUX_VARIANT", "crpo").strip().lower()
EXPLICIT_MODEL = os.environ.get("CT_TANGOFLUX_MODEL", "").strip()
if EXPLICIT_MODEL:
    MODEL_REPO = EXPLICIT_MODEL
    VARIANT = "base" if "base" in EXPLICIT_MODEL.lower() else "crpo"
elif RAW_VARIANT in VARIANT_REPOS:
    MODEL_REPO = VARIANT_REPOS[RAW_VARIANT]
    VARIANT = "base" if "base" in RAW_VARIANT else "crpo"
else:
    MODEL_REPO = "declare-lab/TangoFlux"
    VARIANT = "crpo"

MODEL_ALIAS = os.environ.get("CT_TANGOFLUX_ALIAS", "tangoflux").strip() or "tangoflux"

TF_WORK = Path("/content/tangoflux")
VENV_DIR = TF_WORK / "venv"
SRC_DIR = TF_WORK / "src"
MODELS_ROOT = TF_WORK / "models"
MODEL_DIR = MODELS_ROOT / MODEL_REPO.split("/")[-1]
TEXT_ENCODER_DIR = MODELS_ROOT / "flan-t5-large"
DRIVE_OUTPUT_DIR = Path(DATA_DIR) / "outputs"

TRITON_CACHE_DIR = TF_WORK / "triton-cache"
INDUCTOR_CACHE_DIR = TF_WORK / "inductor-cache"
TRITON_LEGACY_DIR = Path(DATA_DIR) / "triton-cache"
INDUCTOR_LEGACY_DIR = Path(DATA_DIR) / "inductor-cache"

for _d in (TF_WORK, MODELS_ROOT, DRIVE_OUTPUT_DIR, TRITON_CACHE_DIR, INDUCTOR_CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

(TF_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
if str(TF_WORK) not in sys.path:
    sys.path.insert(0, str(TF_WORK))
from colab_cache import restore_cache, save_cache

CACHE_KEY = f"tangoflux-diffusers0303-py{sys.version_info.major}{sys.version_info.minor}-sm{GPU_ARCH}"
CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_PAIRS = (
    (TRITON_CACHE_DIR, TRITON_LEGACY_DIR),
    (INDUCTOR_CACHE_DIR, INDUCTOR_LEGACY_DIR),
)
for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        restore_cache(_local_c, _legacy_c, CACHE_ARCHIVES, CACHE_KEY, tflog)
    except (OSError, ValueError) as err:
        tflog(f"Cache restore warning ({_local_c.name}): {err}")

tflog(f"GPU: {GPU_NAME} ({GPU_VRAM_MIB} MiB, sm_{GPU_ARCH}) | model={MODEL_REPO} ({VARIANT})")

if not shutil.which("ffmpeg"):
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq ffmpeg >/dev/null 2>&1",
        shell=True,
        check=False,
    )
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)

def run_logged(cmd, log_path, env=None, cwd=None, check=True):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(str(x) for x in cmd)}\n")
        f.flush()
        res = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True, timeout=900)
    if check and res.returncode != 0:
        tail = "".join(open(log_path, errors="replace").readlines()[-45:])
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(str(x) for x in cmd[:4])}\n{tail}")
    return res

def prepare_env():
    env_log = TF_WORK / "env-setup.log"
    marker = VENV_DIR / ".ct_tangoflux_env_v2_ready"
    py_bin = VENV_DIR / "bin" / "python"
    if marker.is_file() and py_bin.is_file():
        tflog(f"Reusing existing TangoFlux virtualenv at {VENV_DIR}")
        return py_bin

    if (SRC_DIR / ".git").is_dir():
        run_logged(["git", "-C", str(SRC_DIR), "pull", "--ff-only"], env_log, check=False)
    else:
        shutil.rmtree(SRC_DIR, ignore_errors=True)
        run_logged(["git", "clone", "--depth", "1", TANGOFLUX_GIT_URL, str(SRC_DIR)], env_log)

    tflog(f"Creating TangoFlux venv inheriting Colab PyTorch (log: {env_log})...")
    uv_bin = shutil.which("uv") or "uv"
    run_logged([uv_bin, "venv", "--system-site-packages", str(VENV_DIR), "--clear"], env_log)
    if subprocess.run([str(py_bin), "-c", "import torch"], check=False).returncode != 0:
        run_logged([uv_bin, "pip", "install", "--python", str(py_bin), "torch"], env_log)
    run_logged(
        [
            uv_bin, "pip", "install",
            "--python", str(py_bin),
            "diffusers==0.30.3",
            "transformers>=4.44.2,<4.50.0",
            "safetensors>=0.4.3",
            "huggingface_hub>=0.25.0",
            "hf_xet",
            "sentencepiece>=0.2.0",
            "protobuf>=4.25.0,<7.0.0",
            "psutil",
            "packaging",
            "pyyaml",
            "tqdm",
            "uvicorn>=0.30.0",
            "starlette>=0.38.0",
        ],
        env_log,
    )
    run_logged(
        [uv_bin, "pip", "install", "--python", str(py_bin), "--no-deps", "accelerate>=0.34.2", str(SRC_DIR)],
        env_log,
    )
    check_env = dict(os.environ)
    check_env.update({"USE_TF": "0", "USE_FLAX": "0", "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1"})
    run_logged(
        [
            str(py_bin), "-c",
            "import importlib.machinery, sys, types\n"
            "try:\n"
            "    import datasets\n"
            "except ImportError:\n"
            "    _s = types.ModuleType('datasets')\n"
            "    _s.__spec__ = importlib.machinery.ModuleSpec('datasets', loader=None)\n"
            "    _s.load_dataset = None; _s.Audio = None\n"
            "    sys.modules['datasets'] = _s\n"
            "import torch, diffusers, transformers\n"
            "from tangoflux.model import TangoFlux\n"
            "print('imports-ok', torch.__version__, 'cuda=', torch.cuda.is_available(), diffusers.__version__)",
        ],
        env_log,
        env=check_env,
    )
    marker.write_text("v2\n", encoding="utf-8")
    tflog("TangoFlux virtualenv ready")
    return py_bin

def _cleanup_hf_temp():
    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
        shutil.rmtree(p, ignore_errors=True)
    for mp in (MODEL_DIR / ".cache", TEXT_ENCODER_DIR / ".cache"):
        shutil.rmtree(mp, ignore_errors=True)

def stage_models(py_bin):
    dl_env = dict(os.environ)
    dl_env["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
    dl_env["HF_HUB_DISABLE_TELEMETRY"] = "1"

    def _stage_main():
        if (MODEL_DIR / "tangoflux.safetensors").is_file() and (MODEL_DIR / "vae.safetensors").is_file():
            tflog(f"Reusing local SSD TangoFlux weights at {MODEL_DIR}")
            return
        tflog(f"Downloading {MODEL_REPO} -> {MODEL_DIR} ...")
        subprocess.run(
            [
                str(py_bin), "-c",
                "import os; from huggingface_hub import snapshot_download; "
                f"snapshot_download(repo_id={MODEL_REPO!r}, local_dir={str(MODEL_DIR)!r}, "
                "allow_patterns=['config.json', 'tangoflux.safetensors', 'vae.safetensors'])",
            ],
            env=dl_env,
            check=True,
            timeout=1800,
        )

    def _stage_t5():
        if (TEXT_ENCODER_DIR / "model.safetensors").is_file() and (TEXT_ENCODER_DIR / "spiece.model").is_file():
            tflog(f"Reusing local SSD Flan-T5-Large weights at {TEXT_ENCODER_DIR}")
            return
        tflog(f"Downloading google/flan-t5-large (safetensors only) -> {TEXT_ENCODER_DIR} ...")
        subprocess.run(
            [
                str(py_bin), "-c",
                "from huggingface_hub import snapshot_download; "
                f"snapshot_download(repo_id='google/flan-t5-large', local_dir={str(TEXT_ENCODER_DIR)!r}, "
                "allow_patterns=['*.json', 'spiece.model', 'model.safetensors'])",
            ],
            env=dl_env,
            check=True,
            timeout=1800,
        )

    with ThreadPoolExecutor(max_workers=2) as pool:
        f_main = pool.submit(_stage_main)
        f_t5 = pool.submit(_stage_t5)
        f_main.result()
        f_t5.result()
    _cleanup_hf_temp()

PY_BIN = prepare_env()
stage_models(PY_BIN)
TF_SETUP_OK = True
tflog("Setup complete")


In [ ]:
# @title 2. Service -- launch authenticated TangoFlux ASGI server on 127.0.0.1
from pathlib import Path
if not globals().get("TF_SETUP_OK"):
    raise RuntimeError("TangoFlux setup cell failed before completing model staging; check setup traceback above.")

(TF_WORK / "colab_asgi_common.py").write_text('"""Shared security, Drive path containment, and audio helpers for Colab ASGI middlewares."""\nfrom __future__ import annotations\n\nimport base64\nimport glob as glob_mod\nimport os\nimport re\nimport secrets\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDEFAULT_SKIP_DIRS = {\n    ".git", "node_modules", "__pycache__", ".venv", "venv", "aux-venv", "custom_nodes",\n    "models", ".runtimes", "Takeout", ".Trash", ".ipynb_checkpoints",\n    "flashinfer-cache", "triton-cache", "vllm-cache", "cache-archives", "kernel-archives",\n}\nVOICE_NAME_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")\n\n\ndef validate_voice_name(name: str) -> str:\n    """Validate and normalize a custom voice identifier against path traversal."""\n    vname = (name or "").strip().lower()\n    if not vname or ".." in vname or not VOICE_NAME_RE.fullmatch(vname):\n        raise ValueError(f"Invalid voice name \'{name}\': use 1-64 lowercase [a-z0-9._-] characters without \'..\'")\n    return vname\n\n\ndef _get_header(headers: Any, key: str) -> str:\n    val = headers.get(key)\n    if val:\n        return str(val)\n    if isinstance(headers, dict):\n        low = key.lower()\n        for k, v in headers.items():\n            if str(k).lower() == low and v:\n                return str(v)\n    return ""\n\n\ndef verify_bearer_auth(headers: Any, api_token: str) -> bool:\n    """Fail-closed constant-time Bearer / X-API-Key authentication check."""\n    if not api_token:\n        return False\n    expected_bearer = f"Bearer {api_token}"\n    auth = _get_header(headers, "authorization")\n    if auth and secrets.compare_digest(auth.encode("utf-8"), expected_bearer.encode("utf-8")):\n        return True\n    xkey = _get_header(headers, "x-api-key")\n    if xkey and secrets.compare_digest(xkey.encode("utf-8"), api_token.encode("utf-8")):\n        return True\n    return False\n\n\ndef sanitize_header_param(value: str) -> str:\n    """Strip CR/LF/NUL and escape quotes/backslashes for Content-Disposition parameters."""\n    cleaned = (value or "").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n    return cleaned.replace("\\\\", "\\\\\\\\").replace(\'"\', \'\\\\"\')\n\n\ndef build_multipart_form(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]] | None = None,\n) -> tuple[str, bytes]:\n    """Build a safe multipart/form-data body returning (content_type, body_bytes)."""\n    boundary = f"----ctform{secrets.token_hex(16)}"\n    parts: list[bytes] = []\n    for key, val in fields:\n        safe_key = sanitize_header_param(str(key))\n        parts.append(\n            f\'--{boundary}\\r\\nContent-Disposition: form-data; name="{safe_key}"\\r\\n\\r\\n{val}\\r\\n\'.encode("utf-8")\n        )\n    for field_name, filename, content_type, data in files or []:\n        safe_field = sanitize_header_param(str(field_name))\n        safe_filename = sanitize_header_param(str(filename))\n        safe_ct = (content_type or "application/octet-stream").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n        parts.append(\n            (\n                f"--{boundary}\\r\\n"\n                f\'Content-Disposition: form-data; name="{safe_field}"; filename="{safe_filename}"\\r\\n\'\n                f"Content-Type: {safe_ct}\\r\\n\\r\\n"\n            ).encode("utf-8")\n            + data\n            + b"\\r\\n"\n        )\n    parts.append(f"--{boundary}--\\r\\n".encode("utf-8"))\n    return f"multipart/form-data; boundary={boundary}", b"".join(parts)\n\n\ndef allowed_drive_roots(\n    data_dir: Path,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n) -> tuple[Path, ...]:\n    """Return resolved root directories that Drive/data operations may access."""\n    roots: list[Path] = []\n    if mydrive_root.is_dir():\n        roots.append(mydrive_root.resolve())\n    roots.append(data_dir.resolve())\n    return tuple(dict.fromkeys(roots))\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef resolve_drive_path(\n    raw_spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> Path:\n    """Resolve a Drive or service-data path strictly inside allowed roots."""\n    if "\\x00" in (raw_spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    spec = (raw_spec or "").strip()\n\n    is_file_scheme = spec.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if spec.startswith(prefix):\n            spec = spec[len(prefix):]\n            break\n\n    if is_file_scheme:\n        candidate = Path("/" + spec.lstrip("/"))\n        return _ensure_within_roots(candidate, roots, raw_spec)\n\n    had_leading_slash = spec.startswith("/")\n    clean = spec.lstrip("/")\n    if not clean:\n        if default_subdir and (data_dir / default_subdir).is_dir():\n            return _ensure_within_roots(data_dir / default_subdir, roots, raw_spec)\n        return roots[0]\n\n    if had_leading_slash and not clean.startswith(("content/drive/", "drive/", "MyDrive/", "MyDrive")):\n        # Allow explicit absolute path ONLY if it already lies inside an allowed root (e.g. DATA_DIR in --no-drive mode)\n        return _ensure_within_roots(Path("/" + clean), roots, raw_spec)\n\n    candidates: list[Path] = []\n    if clean.startswith("content/drive/"):\n        candidates.append(Path("/") / clean)\n    elif clean.startswith("drive/"):\n        candidates.append(drive_root.parent / clean)\n    elif clean.startswith("MyDrive/"):\n        candidates.append(drive_root / clean)\n    elif clean == "MyDrive":\n        candidates.append(mydrive_root)\n    elif clean.startswith("colab-llm/") and mydrive_root.is_dir():\n        candidates.append(mydrive_root / clean)\n        candidates.append(data_dir / clean)\n    else:\n        first_seg = clean.split("/", 1)[0]\n        already_prefixed = bool(\n            default_subdir\n            and (\n                clean == default_subdir\n                or clean.startswith(f"{default_subdir}/")\n                or (first_seg != clean and (data_dir / first_seg).is_dir())\n            )\n        )\n        if default_subdir and not already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        candidates.append(data_dir / clean)\n        if default_subdir and already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        if mydrive_root.is_dir():\n            candidates.append(mydrive_root / clean)\n\n    checked: list[Path] = []\n    for cand in candidates:\n        resolved = _ensure_within_roots(cand, roots, raw_spec)\n        checked.append(resolved)\n        if resolved.exists():\n            return resolved\n    if not checked:\n        raise ValueError(f"Path \'{raw_spec}\' could not be resolved inside allowed Drive/data directory")\n    return checked[0]\n\n\ndef rel_to_mydrive(p: Path, mydrive_root: Path = Path("/content/drive/MyDrive")) -> str:\n    try:\n        return str(p.resolve().relative_to(mydrive_root.resolve()))\n    except Exception:\n        return str(p)\n\n\ndef list_drive_audio_files(\n    spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    recursive: bool = True,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List audio files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[Path] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        # Ensure static prefix before wildcard does not escape allowed roots\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots):\n                    continue\n                if hp.is_file() and hp.suffix.lower() in AUDIO_EXTS and str(hp) not in seen:\n                    seen.add(str(hp))\n                    matched_paths.append(hp)\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in AUDIO_EXTS:\n                matched_paths.append(target)\n        elif target.is_dir():\n            if not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if any(resolved.is_relative_to(r) for r in roots) and resolved.is_file() and resolved.suffix.lower() in AUDIO_EXTS:\n                        matched_paths.append(resolved)\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in AUDIO_EXTS:\n                            matched_paths.append(hp)\n\n    items: list[dict[str, Any]] = []\n    for p in matched_paths:\n        st = p.stat()\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef audio_path_to_wav_data_url(\n    audio_path: Path,\n    max_duration_sec: float = 24.0,\n    sample_rate: int = 24000,\n    timeout: int = 60,\n) -> str:\n    """Decode a Drive audio file to a mono PCM16 WAV data URL with bounded ffmpeg timeout."""\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-t", f"{max_duration_sec:.1f}",\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", str(sample_rate), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ]\n    try:\n        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    except OSError as os_err:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {os_err}") from os_err\n    if res.returncode != 0 or not res.stdout:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef save_output_audio_to_drive(\n    audio_bytes: bytes,\n    spec: str,\n    *,\n    data_dir: Path,\n    output_dir: Path,\n    prefix: str,\n    fmt: str = "wav",\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> str:\n    """Persist synthesized audio bytes safely inside Drive or service output directory."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "wav").lower())\n    if not ext or ext == ".":\n        ext = ".wav"\n    if spec and spec not in ("auto", "true", "True", "1"):\n        dest = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            dest = dest / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n    else:\n        dest = output_dir / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n    dest.write_bytes(audio_bytes)\n    return str(dest)\n\n\ndef tail_service_log(\n    log_file: Path = Path("/content/ct/service.log"),\n    lines: int = 120,\n) -> dict[str, Any]:\n    n_lines = max(1, min(int(lines), 2000))\n    entries = log_file.read_text(encoding="utf-8", errors="replace").splitlines()[-n_lines:] if log_file.is_file() else []\n    return {"lines": len(entries), "log": "\\n".join(entries)}\n', encoding="utf-8")
(TF_WORK / "tangoflux_server.py").write_text('"""Standalone ASGI server and security layer for TangoFlux (44.1 kHz stereo text-to-audio / SFX).\n\nImports cleanly with only the Python standard library so local unit tests can exercise all\nauthentication, CORS, Drive path containment, parameter validation, and WAV encoding logic.\n"""\nfrom __future__ import annotations\n\nfrom array import array\nimport asyncio\nimport base64\nfrom concurrent.futures import ThreadPoolExecutor\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport secrets\nimport subprocess\nimport sys\nimport time\nimport types\nfrom typing import Any, Callable\nimport urllib.parse\nimport wave\n\nfrom colab_asgi_common import (\n    list_drive_audio_files,\n    resolve_drive_path,\n    sanitize_header_param,\n    save_output_audio_to_drive,\n    tail_service_log,\n    verify_bearer_auth,\n)\n\nTANGOFLUX_SAMPLE_RATE = 44100\nTANGOFLUX_CHANNELS = 2\nMIN_DURATION_SEC = 1.0\nMAX_DURATION_SEC = 30.0\nDEFAULT_DURATION_SEC = 10.0\nDEFAULT_STEPS = 50\nDEFAULT_GUIDANCE_SCALE = 4.5\nMAX_WAVEFORMS_PER_REQUEST = 4\nMAX_BATCH_ITEMS = 32\nMAX_BODY_BYTES = 8 * 1024 * 1024\n\nAUDIO_MIME_TYPES: dict[str, str] = {\n    "wav": "audio/wav",\n    "mp3": "audio/mpeg",\n    "flac": "audio/flac",\n    "ogg": "audio/ogg",\n    "opus": "audio/ogg",\n    "aac": "audio/aac",\n}\nSUPPORTED_FORMATS = frozenset({*AUDIO_MIME_TYPES.keys(), "json", "b64_json"})\n\nAPI_TOKEN = os.environ.get("CT_API_TOKEN", "")\nSERVED_MODEL = os.environ.get("CT_TANGOFLUX_SERVED_MODEL", "declare-lab/TangoFlux")\nMODEL_ALIAS = os.environ.get("CT_TANGOFLUX_ALIAS", "tangoflux")\nMODEL_VARIANT = os.environ.get("CT_TANGOFLUX_VARIANT", "crpo")\nMODEL_LOCAL_DIR = Path(os.environ.get("CT_TANGOFLUX_MODEL_DIR", "/content/tangoflux/models/tangoflux"))\nTEXT_ENCODER_DIR = Path(os.environ.get("CT_TANGOFLUX_TEXT_ENCODER_DIR", "/content/tangoflux/models/flan-t5-large"))\nDATA_DIR = Path(os.environ.get("CT_DATA_DIR", "/content/drive/MyDrive/colab-llm/tangoflux"))\nDRIVE_ROOT = Path("/content/drive")\nMYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"\nDRIVE_OUTPUT_DIR = DATA_DIR / "outputs"\nLOG_FILE = Path("/content/ct/service.log")\n\nCORS_HEADERS: list[tuple[bytes, bytes]] = [\n    (b"access-control-allow-origin", b"*"),\n    (b"access-control-allow-methods", b"GET, POST, OPTIONS"),\n    (b"access-control-allow-headers", b"Authorization, Content-Type, X-API-Key"),\n    (\n        b"access-control-expose-headers",\n        b"X-Saved-Drive-Path, X-Audio-Sample-Rate, X-Audio-Channels, "\n        b"X-Audio-Duration, X-Audio-Seed, X-Generation-Seconds",\n    ),\n]\n\n\ndef _clamp_sample_i16(val: float) -> int:\n    if not math.isfinite(val):\n        return 0\n    if val >= 1.0:\n        return 32767\n    if val <= -1.0:\n        return -32768\n    return int(round(val * 32767.0))\n\n\ndef float_stereo_to_wav_bytes(\n    channels_first: Any,\n    sample_rate: int = TANGOFLUX_SAMPLE_RATE,\n    headroom_peak: float = 0.98,\n) -> bytes:\n    """Encode a [C, N] or [N] float waveform into 16-bit PCM WAV bytes with peak headroom protection.\n\n    AutoencoderOobleck\'s final linear conv is unbounded and often produces peaks in [1.2, 2.5] on loud\n    SFX prompts (explosions, jet engines, gunfire). Scaling down only when `peak > headroom_peak` matches\n    IPython.display.Audio\'s peak normalization while keeping normal-level waveforms bit-accurate.\n    """\n    if hasattr(channels_first, "detach"):\n        channels_first = channels_first.detach().cpu().float().numpy()\n    try:\n        import numpy as np  # type: ignore[import-not-found]\n\n        arr = np.asarray(channels_first, dtype=np.float32)\n        if arr.ndim == 1:\n            arr = arr[None, :]\n        if arr.ndim != 2:\n            raise ValueError(f"Expected 1D or 2D waveform, got shape {arr.shape}")\n        nch = int(arr.shape[0])\n        clean = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=-1.0)\n        peak = float(np.max(np.abs(clean))) if clean.size else 0.0\n        if peak > headroom_peak:\n            clean = clean * (headroom_peak / peak)\n        clipped = np.clip(clean, -1.0, 1.0)\n        interleaved = (clipped.T * 32767.0).round().astype("<i2")\n        pcm_bytes = interleaved.tobytes(order="C")\n    except ImportError:\n        rows = list(channels_first)\n        if not rows:\n            nch = TANGOFLUX_CHANNELS\n            pcm_bytes = b""\n        elif isinstance(rows[0], (int, float)):\n            nch = 1\n            clean_1d = [0.0 if (math.isnan(float(v)) or math.isinf(float(v))) else float(v) for v in rows]\n            peak = max((abs(v) for v in clean_1d), default=0.0)\n            scale = (headroom_peak / peak) if peak > headroom_peak else 1.0\n            samples = array("h", (_clamp_sample_i16(v * scale) for v in clean_1d))\n            if sys.byteorder != "little":\n                samples.byteswap()\n            pcm_bytes = samples.tobytes()\n        else:\n            nch = len(rows)\n            n_frames = len(rows[0])\n            peak = 0.0\n            for ch in range(nch):\n                for v in rows[ch]:\n                    fv = float(v)\n                    if not (math.isnan(fv) or math.isinf(fv)) and abs(fv) > peak:\n                        peak = abs(fv)\n            scale = (headroom_peak / peak) if peak > headroom_peak else 1.0\n            samples = array("h")\n            for i in range(n_frames):\n                for ch in range(nch):\n                    fv = float(rows[ch][i])\n                    if math.isnan(fv) or math.isinf(fv):\n                        fv = 0.0\n                    samples.append(_clamp_sample_i16(fv * scale))\n            if sys.byteorder != "little":\n                samples.byteswap()\n            pcm_bytes = samples.tobytes()\n\n    buf = io.BytesIO()\n    with wave.open(buf, "wb") as wf:\n        wf.setnchannels(max(1, nch))\n        wf.setsampwidth(2)\n        wf.setframerate(int(sample_rate))\n        wf.writeframes(pcm_bytes)\n    return buf.getvalue()\n\n\ndef transcode_audio_bytes(wav_bytes: bytes, fmt: str, timeout: int = 60) -> bytes:\n    """Convert 44.1 kHz stereo WAV bytes to compressed audio formats using ffmpeg."""\n    target = (fmt or "wav").lower().strip()\n    if target == "wav":\n        return wav_bytes\n    codec_map: dict[str, tuple[str, list[str]]] = {\n        "mp3": ("mp3", ["-c:a", "libmp3lame", "-q:a", "2"]),\n        "flac": ("flac", ["-c:a", "flac"]),\n        "ogg": ("ogg", ["-c:a", "libvorbis", "-q:a", "5"]),\n        "opus": ("ogg", ["-c:a", "libopus", "-b:a", "128k"]),\n        "aac": ("adts", ["-c:a", "aac", "-b:a", "160k"]),\n    }\n    if target not in codec_map:\n        raise ValueError(f"Unsupported output audio format \'{fmt}\'")\n    muxer, codec_args = codec_map[target]\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-f", "wav", "-i", "pipe:0",\n        "-vn", *codec_args, "-f", muxer, "pipe:1",\n    ]\n    res = subprocess.run(cmd, input=wav_bytes, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    if res.returncode != 0 or not res.stdout:\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed encoding {target}: {err}")\n    return res.stdout\n\n\ndef _validate_save_spec(save_spec: Any) -> str | None:\n    if save_spec is None or save_spec is False or save_spec == "":\n        return None\n    spec_str = "auto" if save_spec is True else str(save_spec).strip()\n    if not spec_str:\n        return None\n    if spec_str not in ("auto", "true", "True", "1"):\n        resolve_drive_path(\n            spec_str,\n            data_dir=DATA_DIR,\n            default_subdir="outputs",\n            mydrive_root=MYDRIVE_ROOT,\n            drive_root=DRIVE_ROOT,\n        )\n    return spec_str\n\n\ndef _parse_generate_params(data: dict[str, Any]) -> dict[str, Any]:\n    raw_prompt = data.get("prompt")\n    if raw_prompt is None:\n        raw_prompt = data.get("input")\n    if not isinstance(raw_prompt, str) or not raw_prompt.strip():\n        raise ValueError("Provide a non-empty string in `prompt` (or `input`).")\n    prompt = raw_prompt.strip()\n    if len(prompt) > 2000:\n        raise ValueError("`prompt` must be at most 2000 characters.")\n\n    raw_dur = data.get("duration", DEFAULT_DURATION_SEC)\n    try:\n        duration = float(raw_dur)\n    except (TypeError, ValueError):\n        raise ValueError("`duration` must be a number between 1.0 and 30.0 seconds.") from None\n    if not math.isfinite(duration) or duration < MIN_DURATION_SEC or duration > MAX_DURATION_SEC:\n        raise ValueError(f"`duration` must be between {MIN_DURATION_SEC:g} and {MAX_DURATION_SEC:g} seconds (got {raw_dur!r}).")\n\n    raw_steps = data.get("steps", data.get("num_inference_steps", DEFAULT_STEPS))\n    try:\n        steps = int(raw_steps)\n    except (TypeError, ValueError):\n        raise ValueError("`steps` must be an integer between 1 and 200.") from None\n    if steps < 1 or steps > 200:\n        raise ValueError(f"`steps` must be between 1 and 200 (got {steps}).")\n\n    raw_cfg = data.get("guidance_scale", data.get("cfg_scale", DEFAULT_GUIDANCE_SCALE))\n    try:\n        guidance_scale = float(raw_cfg)\n    except (TypeError, ValueError):\n        raise ValueError("`guidance_scale` must be a number between 1.0 and 20.0.") from None\n    if not math.isfinite(guidance_scale) or guidance_scale < 1.0 or guidance_scale > 20.0:\n        raise ValueError(f"`guidance_scale` must be between 1.0 and 20.0 (got {raw_cfg!r}).")\n\n    raw_n = data.get("num_waveforms", data.get("n", 1))\n    try:\n        num_waveforms = int(raw_n)\n    except (TypeError, ValueError):\n        raise ValueError(f"`num_waveforms` must be an integer between 1 and {MAX_WAVEFORMS_PER_REQUEST}.") from None\n    if num_waveforms < 1 or num_waveforms > MAX_WAVEFORMS_PER_REQUEST:\n        raise ValueError(f"`num_waveforms` must be between 1 and {MAX_WAVEFORMS_PER_REQUEST} (got {num_waveforms}).")\n\n    raw_seed = data.get("seed")\n    if raw_seed is None or raw_seed == "" or (isinstance(raw_seed, int) and raw_seed < 0):\n        seed = secrets.randbelow(2**31 - 1)\n    else:\n        try:\n            seed = int(raw_seed)\n        except (TypeError, ValueError):\n            raise ValueError("`seed` must be a non-negative integer.") from None\n        if seed < 0 or seed > 2**32 - 1:\n            raise ValueError("`seed` must be between 0 and 4294967295.")\n\n    raw_fmt = str(data.get("response_format") or data.get("format") or "wav").strip().lower()\n    if raw_fmt not in SUPPORTED_FORMATS:\n        raise ValueError(f"Unsupported `response_format` \'{raw_fmt}\'; choose from {sorted(SUPPORTED_FORMATS)}.")\n    return_json = bool(data.get("return_json")) or raw_fmt in ("json", "b64_json") or num_waveforms > 1\n    audio_fmt = "wav" if raw_fmt in ("json", "b64_json") else raw_fmt\n\n    save_spec = _validate_save_spec(data.get("save_to_drive"))\n    return {\n        "prompt": prompt,\n        "duration": duration,\n        "steps": steps,\n        "guidance_scale": guidance_scale,\n        "num_waveforms": num_waveforms,\n        "seed": seed,\n        "audio_format": audio_fmt,\n        "return_json": return_json,\n        "save_to_drive": save_spec,\n    }\n\n\ndef _prepare_generate_request(raw_body: bytes) -> dict[str, Any]:\n    if not raw_body:\n        raise ValueError("Request JSON body is required.")\n    try:\n        data = json.loads(raw_body.decode("utf-8"))\n    except Exception as err:\n        raise ValueError(f"Invalid JSON body: {err}") from None\n    if not isinstance(data, dict):\n        raise ValueError("Request JSON body must be an object.")\n    return _parse_generate_params(data)\n\n\ndef _prepare_batch_request(raw_body: bytes) -> dict[str, Any]:\n    if not raw_body:\n        raise ValueError("Request JSON body is required.")\n    try:\n        data = json.loads(raw_body.decode("utf-8"))\n    except Exception as err:\n        raise ValueError(f"Invalid JSON body: {err}") from None\n    if not isinstance(data, dict):\n        raise ValueError("Batch request JSON body must be an object.")\n    items = data.get("items")\n    if not isinstance(items, list) or not items:\n        raise ValueError("Batch request must provide a non-empty `items` list.")\n    if len(items) > MAX_BATCH_ITEMS:\n        raise ValueError(f"`items` exceeds maximum batch size of {MAX_BATCH_ITEMS}.")\n\n    default_fmt = str(data.get("response_format") or data.get("format") or "wav").strip().lower()\n    if default_fmt in ("json", "b64_json"):\n        default_fmt = "wav"\n    if default_fmt not in AUDIO_MIME_TYPES:\n        raise ValueError(f"Unsupported batch `response_format` \'{default_fmt}\'.")\n\n    top_save = _validate_save_spec(data.get("save_to_drive"))\n    parsed_items: list[dict[str, Any]] = []\n    for idx, entry in enumerate(items):\n        if isinstance(entry, str):\n            entry = {"prompt": entry}\n        if not isinstance(entry, dict):\n            raise ValueError(f"Batch item {idx} must be an object or prompt string.")\n        merged = {\n            "duration": data.get("duration", DEFAULT_DURATION_SEC),\n            "steps": data.get("steps", DEFAULT_STEPS),\n            "guidance_scale": data.get("guidance_scale", DEFAULT_GUIDANCE_SCALE),\n            "response_format": entry.get("response_format") or entry.get("format") or default_fmt,\n            **entry,\n            "num_waveforms": 1,\n        }\n        if "save_to_drive" not in merged and top_save:\n            merged["save_to_drive"] = top_save\n        spec = _parse_generate_params(merged)\n        spec["name"] = str(entry.get("name") or f"sfx-{idx + 1:03d}").strip()[:80]\n        parsed_items.append(spec)\n    return {"items": parsed_items, "response_format": default_fmt}\n\n\nclass TangoFluxEngine:\n    """GPU inference wrapper around declare-lab/TangoFlux with local SSD loading and multi-sample decoding."""\n\n    def __init__(self, model_dir: Path, text_encoder_dir: Path | None = None, device: str | None = None):\n        for _tf_flag, _tf_val in (\n            ("USE_TF", "0"),\n            ("USE_FLAX", "0"),\n            ("TRANSFORMERS_NO_TF", "1"),\n            ("TRANSFORMERS_NO_FLAX", "1"),\n        ):\n            os.environ.setdefault(_tf_flag, _tf_val)\n\n        # Upstream tangoflux/model.py imports unused `datasets` at module level; stub if absent so pyarrow isn\'t needed.\n        try:\n            import datasets  # noqa: F401\n        except ImportError:\n            import importlib.machinery\n\n            stub = types.ModuleType("datasets")\n            stub.__spec__ = importlib.machinery.ModuleSpec("datasets", loader=None)\n            stub.load_dataset = None  # type: ignore[attr-defined]\n            stub.Audio = None  # type: ignore[attr-defined]\n            sys.modules["datasets"] = stub\n\n        import torch  # type: ignore[import-not-found]\n        from diffusers import AutoencoderOobleck  # type: ignore[import-not-found]\n        from safetensors.torch import load_file  # type: ignore[import-not-found]\n        from tangoflux.model import TangoFlux  # type: ignore[import-not-found]\n\n        self.torch = torch\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        self.use_cuda_amp = bool(self.device.startswith("cuda") and torch.cuda.is_available())\n        if self.use_cuda_amp:\n            torch.backends.cuda.matmul.allow_tf32 = True\n            torch.backends.cudnn.allow_tf32 = True\n            self.gpu_name = torch.cuda.get_device_name(0)\n            bf16_ok = bool(hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())\n            self.amp_dtype = torch.bfloat16 if bf16_ok else torch.float16\n        else:\n            self.gpu_name = "cpu"\n            self.amp_dtype = torch.float32\n\n        cfg_path = model_dir / "config.json"\n        config = json.loads(cfg_path.read_text(encoding="utf-8"))\n        enc_arg = str(text_encoder_dir) if (text_encoder_dir and text_encoder_dir.is_dir()) else None\n\n        self.vae = AutoencoderOobleck()\n        self.vae.load_state_dict(load_file(str(model_dir / "vae.safetensors")))\n        self.model = TangoFlux(config, text_encoder_dir=enc_arg)\n        self.model.load_state_dict(load_file(str(model_dir / "tangoflux.safetensors")), strict=False)\n\n        # Keep Flan-T5-Large in full FP32 (avoids FP16 overflow NaNs) and fix upstream bug where\n        # guidance_scale <= 1.0 passes unsupported `num_samples_per_prompt` to `encode_text`.\n        orig_encode_text = self.model.encode_text\n\n        def _encode_text_compat(prompt: Any, num_samples_per_prompt: int = 1):\n            if self.use_cuda_amp:\n                with torch.amp.autocast("cuda", enabled=False):\n                    enc, mask = orig_encode_text(prompt)\n            else:\n                enc, mask = orig_encode_text(prompt)\n            if num_samples_per_prompt > 1:\n                enc = enc.repeat_interleave(num_samples_per_prompt, 0)\n                mask = mask.repeat_interleave(num_samples_per_prompt, 0)\n            return enc, mask\n\n        self.model.encode_text = _encode_text_compat  # type: ignore[method-assign]\n\n        self.vae.to(self.device).eval()\n        self.model.to(self.device).eval()\n        self.sample_rate = int(getattr(self.vae.config, "sampling_rate", TANGOFLUX_SAMPLE_RATE))\n\n    def generate_waveforms(\n        self,\n        *,\n        prompt: str,\n        duration: float,\n        steps: int,\n        guidance_scale: float,\n        seed: int,\n        num_waveforms: int = 1,\n    ) -> list[Any]:\n        torch = self.torch\n        with torch.no_grad():\n            if self.use_cuda_amp:\n                with torch.amp.autocast("cuda", dtype=self.amp_dtype):\n                    latents = self.model.inference_flow(\n                        prompt,\n                        duration=float(duration),\n                        num_inference_steps=int(steps),\n                        guidance_scale=float(guidance_scale),\n                        seed=int(seed),\n                        disable_progress=True,\n                        num_samples_per_prompt=int(num_waveforms),\n                    )\n                latents = latents.float()\n            else:\n                latents = self.model.inference_flow(\n                    prompt,\n                    duration=float(duration),\n                    num_inference_steps=int(steps),\n                    guidance_scale=float(guidance_scale),\n                    seed=int(seed),\n                    disable_progress=True,\n                    num_samples_per_prompt=int(num_waveforms),\n                )\n            decoded = self.vae.decode(latents.transpose(2, 1)).sample.float().cpu()\n        waveform_end = max(1, int(round(float(duration) * self.sample_rate)))\n        return [decoded[i, :, :waveform_end] for i in range(int(decoded.shape[0]))]\n\n\ndef _headers_to_dict(raw_headers: list[tuple[bytes, bytes]]) -> dict[str, str]:\n    out: dict[str, str] = {}\n    for k, v in raw_headers:\n        ks = k.decode("latin1").lower()\n        vs = v.decode("latin1")\n        if ks not in out:\n            out[ks] = vs\n    return out\n\n\nasync def _read_body(receive: Any, max_bytes: int = MAX_BODY_BYTES) -> bytes:\n    chunks: list[bytes] = []\n    total = 0\n    while True:\n        msg = await receive()\n        if msg["type"] == "http.disconnect":\n            break\n        piece = msg.get("body", b"")\n        if piece:\n            total += len(piece)\n            if total > max_bytes:\n                raise ValueError(f"Request body exceeds {max_bytes // (1024 * 1024)} MiB limit")\n            chunks.append(piece)\n        if not msg.get("more_body", False):\n            break\n    return b"".join(chunks)\n\n\nasync def _send_json(\n    send: Any,\n    status: int,\n    payload: Any,\n    extra_headers: list[tuple[bytes, bytes]] | None = None,\n) -> None:\n    raw = json.dumps(payload, ensure_ascii=False).encode("utf-8")\n    hdrs = [\n        *CORS_HEADERS,\n        (b"content-type", b"application/json; charset=utf-8"),\n        (b"content-length", str(len(raw)).encode("ascii")),\n        *(extra_headers or []),\n    ]\n    await send({"type": "http.response.start", "status": status, "headers": hdrs})\n    await send({"type": "http.response.body", "body": raw, "more_body": False})\n\n\nasync def _send_bytes(\n    send: Any,\n    status: int,\n    body: bytes,\n    content_type: str,\n    extra_headers: list[tuple[bytes, bytes]] | None = None,\n) -> None:\n    hdrs = [\n        *CORS_HEADERS,\n        (b"content-type", content_type.encode("latin1")),\n        (b"content-length", str(len(body)).encode("ascii")),\n        *(extra_headers or []),\n    ]\n    await send({"type": "http.response.start", "status": status, "headers": hdrs})\n    await send({"type": "http.response.body", "body": body, "more_body": False})\n\n\nclass TangoFluxApp:\n    """Authenticated ASGI application serving TangoFlux sound-effect generation and Drive I/O."""\n\n    def __init__(self, generate_fn: Callable[..., list[Any]], *, gpu_name: str = "cuda", sample_rate: int = TANGOFLUX_SAMPLE_RATE):\n        self._generate_fn = generate_fn\n        self._gpu_name = gpu_name\n        self._sample_rate = sample_rate\n        self._gpu_lock = asyncio.Lock()\n        self._executor = ThreadPoolExecutor(max_workers=1)\n\n    def _synthesize_encoded(self, spec: dict[str, Any], prefix: str = "tangoflux") -> dict[str, Any]:\n        t0 = time.monotonic()\n        waves = self._generate_fn(\n            prompt=spec["prompt"],\n            duration=spec["duration"],\n            steps=spec["steps"],\n            guidance_scale=spec["guidance_scale"],\n            seed=spec["seed"],\n            num_waveforms=spec["num_waveforms"],\n        )\n        elapsed = time.monotonic() - t0\n        fmt = spec["audio_format"]\n        save_spec = spec.get("save_to_drive")\n\n        items: list[dict[str, Any]] = []\n        for idx, w in enumerate(waves):\n            wav_bytes = float_stereo_to_wav_bytes(w, sample_rate=self._sample_rate)\n            encoded = transcode_audio_bytes(wav_bytes, fmt) if fmt != "wav" else wav_bytes\n            saved_path: str | None = None\n            if save_spec:\n                item_save = save_spec\n                if len(waves) > 1 and save_spec not in ("auto", "true", "True", "1") and not save_spec.endswith("/"):\n                    p_obj = Path(save_spec)\n                    stem_s = f"{p_obj.stem}_{idx + 1:02d}{p_obj.suffix}"\n                    item_save = str(p_obj.with_name(stem_s))\n                saved_path = save_output_audio_to_drive(\n                    encoded,\n                    item_save,\n                    data_dir=DATA_DIR,\n                    output_dir=DRIVE_OUTPUT_DIR,\n                    prefix=f"{prefix}-{idx + 1:02d}" if len(waves) > 1 else prefix,\n                    fmt=fmt,\n                    mydrive_root=MYDRIVE_ROOT,\n                    drive_root=DRIVE_ROOT,\n                )\n            items.append({\n                "index": idx,\n                "audio_bytes": encoded,\n                "format": fmt,\n                "saved_drive_path": saved_path,\n            })\n        return {\n            "elapsed_sec": round(elapsed, 3),\n            "items": items,\n        }\n\n    async def __call__(self, scope: dict[str, Any], receive: Any, send: Any) -> None:\n        if scope["type"] != "http":\n            return\n\n        method = scope.get("method", "GET").upper()\n        path = scope.get("path", "/")\n        if method == "OPTIONS":\n            await send({"type": "http.response.start", "status": 204, "headers": list(CORS_HEADERS)})\n            await send({"type": "http.response.body", "body": b"", "more_body": False})\n            return\n\n        headers = _headers_to_dict(scope.get("headers") or [])\n        if not verify_bearer_auth(headers, API_TOKEN):\n            await _send_json(\n                send,\n                401,\n                {"error": {"message": "Missing or invalid Bearer API token", "type": "authentication_error"}},\n                extra_headers=[(b"www-authenticate", b\'Bearer realm="tangoflux"\')],\n            )\n            return\n\n        qs = urllib.parse.parse_qs((scope.get("query_string") or b"").decode("latin1"))\n\n        if method == "GET" and path == "/health":\n            await _send_json(send, 200, {\n                "status": "ok",\n                "ready": True,\n                "model": SERVED_MODEL,\n                "alias": MODEL_ALIAS,\n                "sample_rate": self._sample_rate,\n                "channels": TANGOFLUX_CHANNELS,\n            })\n            return\n\n        if method == "GET" and path == "/v1/models":\n            created = int(time.time())\n            models = [\n                {"id": MODEL_ALIAS, "object": "model", "created": created, "owned_by": "declare-lab"},\n            ]\n            if SERVED_MODEL != MODEL_ALIAS:\n                models.append({"id": SERVED_MODEL, "object": "model", "created": created, "owned_by": "declare-lab"})\n            await _send_json(send, 200, {"object": "list", "data": models})\n            return\n\n        if method == "GET" and path == "/api/ct/info":\n            await _send_json(send, 200, {\n                "service": "tangoflux",\n                "model": SERVED_MODEL,\n                "alias": MODEL_ALIAS,\n                "variant": MODEL_VARIANT,\n                "gpu_name": self._gpu_name,\n                "sample_rate": self._sample_rate,\n                "channels": TANGOFLUX_CHANNELS,\n                "duration_range_sec": [MIN_DURATION_SEC, MAX_DURATION_SEC],\n                "default_duration_sec": DEFAULT_DURATION_SEC,\n                "default_steps": DEFAULT_STEPS,\n                "default_guidance_scale": DEFAULT_GUIDANCE_SCALE,\n                "data_dir": str(DATA_DIR),\n                "drive_output_dir": str(DRIVE_OUTPUT_DIR),\n            })\n            return\n\n        if method == "GET" and path == "/api/ct/logs":\n            n_lines = int((qs.get("lines") or ["120"])[0])\n            await _send_json(send, 200, tail_service_log(LOG_FILE, n_lines))\n            return\n\n        if method == "GET" and path == "/api/ct/drive-list":\n            spec = (qs.get("path") or [""])[0]\n            recursive = (qs.get("recursive") or ["1"])[0] not in ("0", "false", "False")\n            try:\n                files = list_drive_audio_files(\n                    spec,\n                    data_dir=DATA_DIR,\n                    default_subdir="outputs",\n                    recursive=recursive,\n                    mydrive_root=MYDRIVE_ROOT,\n                    drive_root=DRIVE_ROOT,\n                )\n                await _send_json(send, 200, {"count": len(files), "files": files})\n            except ValueError as err:\n                await _send_json(send, 400, {"error": {"message": str(err), "type": "invalid_request_error"}})\n            except Exception as err:\n                await _send_json(send, 500, {"error": {"message": str(err), "type": "server_error"}})\n            return\n\n        if method == "POST" and path in ("/api/ct/generate", "/v1/audio/generations", "/v1/audio/speech"):\n            try:\n                raw_body = await _read_body(receive)\n                spec = _prepare_generate_request(raw_body)\n            except ValueError as err:\n                await _send_json(send, 400, {"error": {"message": str(err), "type": "invalid_request_error"}})\n                return\n\n            try:\n                loop = asyncio.get_running_loop()\n                async with self._gpu_lock:\n                    gen_out = await loop.run_in_executor(self._executor, self._synthesize_encoded, spec, "tangoflux")\n            except ValueError as err:\n                await _send_json(send, 400, {"error": {"message": str(err), "type": "invalid_request_error"}})\n                return\n            except Exception as err:\n                await _send_json(send, 500, {"error": {"message": str(err), "type": "generation_error"}})\n                return\n\n            out_items = gen_out["items"]\n            if not spec["return_json"] and len(out_items) == 1:\n                item0 = out_items[0]\n                resp_hdrs: list[tuple[bytes, bytes]] = [\n                    (b"x-audio-sample-rate", str(self._sample_rate).encode("ascii")),\n                    (b"x-audio-channels", str(TANGOFLUX_CHANNELS).encode("ascii")),\n                    (b"x-audio-duration", f"{spec[\'duration\']:.2f}".encode("ascii")),\n                    (b"x-audio-seed", str(spec["seed"]).encode("ascii")),\n                    (b"x-generation-seconds", str(gen_out["elapsed_sec"]).encode("ascii")),\n                ]\n                if item0.get("saved_drive_path"):\n                    safe_dp = sanitize_header_param(str(item0["saved_drive_path"]))\n                    resp_hdrs.append((b"x-saved-drive-path", safe_dp.encode("latin1", errors="replace")))\n                await _send_bytes(\n                    send,\n                    200,\n                    item0["audio_bytes"],\n                    AUDIO_MIME_TYPES[item0["format"]],\n                    extra_headers=resp_hdrs,\n                )\n                return\n\n            payload = {\n                "id": f"sfx-{secrets.token_hex(6)}",\n                "model": SERVED_MODEL,\n                "prompt": spec["prompt"],\n                "duration": spec["duration"],\n                "steps": spec["steps"],\n                "guidance_scale": spec["guidance_scale"],\n                "seed": spec["seed"],\n                "sample_rate": self._sample_rate,\n                "channels": TANGOFLUX_CHANNELS,\n                "elapsed_sec": gen_out["elapsed_sec"],\n                "data": [\n                    {\n                        "index": it["index"],\n                        "format": it["format"],\n                        "media_type": AUDIO_MIME_TYPES[it["format"]],\n                        "b64_json": base64.b64encode(it["audio_bytes"]).decode("ascii"),\n                        "saved_drive_path": it.get("saved_drive_path"),\n                    }\n                    for it in out_items\n                ],\n            }\n            await _send_json(send, 200, payload)\n            return\n\n        if method == "POST" and path == "/api/ct/batch":\n            try:\n                raw_body = await _read_body(receive)\n                batch_spec = _prepare_batch_request(raw_body)\n            except ValueError as err:\n                await _send_json(send, 400, {"error": {"message": str(err), "type": "invalid_request_error"}})\n                return\n\n            results: list[dict[str, Any]] = []\n            t_batch = time.monotonic()\n            loop = asyncio.get_running_loop()\n            async with self._gpu_lock:\n                for idx, item_spec in enumerate(batch_spec["items"]):\n                    safe_prefix = "".join(\n                        ch if (ch.isalnum() or ch in "-_") else "-"\n                        for ch in item_spec.get("name", f"sfx-{idx + 1:03d}")\n                    ).strip("-") or f"sfx-{idx + 1:03d}"\n                    try:\n                        gen_out = await loop.run_in_executor(\n                            self._executor, self._synthesize_encoded, item_spec, safe_prefix[:48]\n                        )\n                        it0 = gen_out["items"][0]\n                        results.append({\n                            "index": idx,\n                            "name": item_spec["name"],\n                            "status": "success",\n                            "prompt": item_spec["prompt"],\n                            "duration": item_spec["duration"],\n                            "steps": item_spec["steps"],\n                            "guidance_scale": item_spec["guidance_scale"],\n                            "seed": item_spec["seed"],\n                            "format": it0["format"],\n                            "media_type": AUDIO_MIME_TYPES[it0["format"]],\n                            "elapsed_sec": gen_out["elapsed_sec"],\n                            "audio_data": base64.b64encode(it0["audio_bytes"]).decode("ascii"),\n                            "saved_drive_path": it0.get("saved_drive_path"),\n                        })\n                    except Exception as err:\n                        results.append({\n                            "index": idx,\n                            "name": item_spec["name"],\n                            "status": "error",\n                            "prompt": item_spec["prompt"],\n                            "error": str(err),\n                        })\n\n            await _send_json(send, 200, {\n                "id": f"batch-{secrets.token_hex(6)}",\n                "model": SERVED_MODEL,\n                "sample_rate": self._sample_rate,\n                "channels": TANGOFLUX_CHANNELS,\n                "elapsed_sec": round(time.monotonic() - t_batch, 3),\n                "results": results,\n            })\n            return\n\n        await _send_json(send, 404, {"error": {"message": f"Unknown endpoint {method} {path}", "type": "not_found"}})\n\n\ndef run_tangoflux_server(host: str = "127.0.0.1", port: int = 8000) -> None:\n    import uvicorn  # type: ignore[import-not-found]\n\n    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    print(f"[tangoflux] Loading {SERVED_MODEL} from {MODEL_LOCAL_DIR} ...", flush=True)\n    t0 = time.monotonic()\n    engine = TangoFluxEngine(model_dir=MODEL_LOCAL_DIR, text_encoder_dir=TEXT_ENCODER_DIR)\n    print(\n        f"[tangoflux] Model loaded on {engine.device} ({engine.gpu_name}) in {time.monotonic() - t0:.1f}s "\n        f"(sample_rate={engine.sample_rate} Hz, stereo)",\n        flush=True,\n    )\n    app = TangoFluxApp(engine.generate_waveforms, gpu_name=engine.gpu_name, sample_rate=engine.sample_rate)\n    uvicorn.run(app, host=host, port=int(port), log_level="info")\n\n\nif __name__ == "__main__":\n    run_tangoflux_server(\n        host="127.0.0.1",\n        port=int(os.environ.get("CT_SERVICE_PORT", "8000")),\n    )\n', encoding="utf-8")

def start_service():
    if not API_TOKEN:
        raise RuntimeError("CT_API_TOKEN is empty; refusing to start unauthenticated TangoFlux server")
    if "service" in PROCS:
        ct_stop_proc(PROCS["service"][0])
    subprocess.run(["pkill", "-9", "-f", "tangoflux_server.py"], check=False)
    time.sleep(1)
    svc_env = dict(os.environ)
    svc_env.update({
        "CT_API_TOKEN": API_TOKEN,
        "CT_SERVICE_PORT": str(SERVICE_PORT),
        "CT_DATA_DIR": str(DATA_DIR),
        "CT_TANGOFLUX_SERVED_MODEL": MODEL_REPO,
        "CT_TANGOFLUX_ALIAS": MODEL_ALIAS,
        "CT_TANGOFLUX_VARIANT": VARIANT,
        "CT_TANGOFLUX_MODEL_DIR": str(MODEL_DIR),
        "CT_TANGOFLUX_TEXT_ENCODER_DIR": str(TEXT_ENCODER_DIR),
        "TRITON_CACHE_DIR": str(TRITON_CACHE_DIR),
        "TORCHINDUCTOR_CACHE_DIR": str(INDUCTOR_CACHE_DIR),
        "USE_TF": "0",
        "USE_FLAX": "0",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_FLAX": "1",
        "PYTHONPATH": f"{TF_WORK}:{svc_env.get('PYTHONPATH', '')}".rstrip(":"),
    })
    proc = ct_spawn(
        "service",
        [str(VENV_DIR / "bin" / "python"), str(TF_WORK / "tangoflux_server.py")],
        env=svc_env,
        cwd=str(TF_WORK),
    )
    PROCS["service"] = (proc, start_service)
    return proc

start_service()
tflog(f"Spawned TangoFlux server (pid={PROCS['service'][0].pid}) on 127.0.0.1:{SERVICE_PORT}")


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Readiness gate, warmup SFX generation, cache archiving & watchdog
import urllib.error
import urllib.request
if not globals().get("TF_SETUP_OK") or "service" not in PROCS:
    raise RuntimeError("TangoFlux service did not start because an earlier notebook cell failed; see traceback above.")

def http_json(url, token="", payload=None, timeout=20):
    headers = {"User-Agent": "Mozilla/5.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    data = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers=headers, method="POST" if data is not None else "GET")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = r.read().decode("utf-8", errors="replace")
            return r.status, (json.loads(body) if body.strip().startswith("{") else {})
    except urllib.error.HTTPError as e:
        return e.code, {}

def auth_check(base, token):
    for candidate, expected in (("", 401), ("invalid-probe-token", 401), (token, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Auth test failed on {base}/v1/models: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 1800
last_notice = 0
while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("TangoFlux server exited during startup:\n" + ct_tail("service", 90))
    try:
        code, _ = http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)
        if code == 200:
            break
    except Exception:
        pass
    if time.monotonic() - last_notice >= 20:
        last_line = (ct_tail("service", 3).strip().splitlines() or ["loading weights..."])[-1]
        tflog(f"Waiting for TangoFlux server | GPU: {ct_gpu_stat()} | {last_line[:100]}")
        last_notice = time.monotonic()
    time.sleep(4)
else:
    raise RuntimeError("Timed out waiting for TangoFlux server to become ready:\n" + ct_tail("service", 90))

auth_check(LOCAL_BASE, API_TOKEN)

# Warmup generation to compile/cache CUDA & Triton kernels before opening public gate
warmup_req = urllib.request.Request(
    LOCAL_BASE + "/api/ct/generate",
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_TOKEN}"},
    data=json.dumps({
        "prompt": "Gentle rain falling on wooden roof",
        "duration": 2.0,
        "steps": 10,
        "guidance_scale": 4.5,
        "seed": 0,
        "response_format": "wav",
    }).encode("utf-8"),
)
try:
    with urllib.request.urlopen(warmup_req, timeout=180) as wresp:
        wav_bytes = wresp.read()
        tflog(f"Warmup SFX generation succeeded ({len(wav_bytes):,} bytes 44.1 kHz stereo WAV)")
except Exception as werr:
    tflog(f"Warmup SFX generation warning: {werr}")

for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        save_cache(_local_c, CACHE_ARCHIVES, CACHE_KEY, tflog)
    except (OSError, ValueError) as err:
        tflog(f"Cache archive save warning ({_local_c.name}): {err}")

ct_public_gate(
    PUBLIC_URL,
    lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
    lambda: auth_check(PUBLIC_URL, API_TOKEN),
    tflog,
    timeout=180,
    interval=8,
)
tflog(f"Public endpoint verified: {PUBLIC_URL}/v1/models")

print(f"{SERVICE} is up: {PUBLIC_URL} (model={MODEL_REPO})")
print("CT_READY", flush=True)

ct_watchdog(
    lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200,
    start_service,
    tflog,
    heartbeat=lambda up: tflog(f"heartbeat | up={up}s | GPU: {ct_gpu_stat()}"),
)
